# Network Module — Anomaly Detection (Isolation Forest)

CM3070 Final Year Project — Phase 4. Trains an unsupervised Isolation Forest on
the full CICIDS2017 Wednesday file and saves the model as a joblib bundle for
the FastAPI inference service. The model learns from benign flows only. All
non-benign labels are combined into one attack class for evaluation.

Dataset: CICIDS2017 Wednesday file, 692,703 rows, 78 flow features, 63.5%
benign / 36.5% attack, and 1,008 missing feature values before cleaning.
Labels: BENIGN 440,031; DoS Hulk 231,073; DoS GoldenEye 10,293; DoS
slowloris 5,796; DoS Slowhttptest 5,499; Heartbleed 11.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install scikit-learn==1.6.1 joblib pandas numpy -q

In [ ]:
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

In [ ]:
# Configuration
CONFIG = {
    "csv_path": (
        "/content/drive/MyDrive/cm3070_datasets/"
        "Wednesday-workingHours.pcap_ISCX.csv"
    ),
    "test_ratio": 0.20,
    "random_seed": 42,
    "n_estimators": 200,
    "contamination": "auto",
    "output_path": "/content/network_model.joblib",
}

# The label value that marks normal traffic; everything else is an attack.
BENIGN_LABEL = "BENIGN"
MEDIUM_SEVERITY_THRESHOLD = 0.40
HIGH_SEVERITY_THRESHOLD = 0.70
EVALUATION_THRESHOLDS = [
    0.30,
    MEDIUM_SEVERITY_THRESHOLD,
    0.50,
    HIGH_SEVERITY_THRESHOLD,
]
MINIMUM_EXPECTED_ROC_AUC = 0.70

In [ ]:
# Load the dataset
print("Loading dataset...")
network_data = pd.read_csv(CONFIG["csv_path"])

# Every column name has a leading space in this dataset.
network_data.columns = network_data.columns.str.strip()

print(f"Total rows:  {len(network_data):,}")
print(f"Columns:     {len(network_data.columns)}")
print("Label balance:")
print(network_data["Label"].value_counts())
print(f"Nulls:       {network_data.isnull().sum().sum()}")

# Remove unusable rows if a different dataset copy contains them.
network_data = network_data.dropna(how="all")
network_data = network_data.dropna(subset=["Label"])

In [ ]:
# Prepare features and the evaluation label
feature_names = [
    column for column in network_data.columns if column != "Label"
]
network_features = network_data[feature_names].copy()

# CICIDS2017 ratio features contain inf/NaN, so use the inference cleaning rule.
network_features = network_features.replace(
    [np.inf, -np.inf],
    np.nan,
).fillna(0.0)

# Labels are kept separate because the model uses them only for evaluation.
attack_labels = (
    network_data["Label"] != BENIGN_LABEL
).astype(int).to_numpy()
feature_matrix = network_features.to_numpy(dtype=float)

print(f"Feature matrix: {feature_matrix.shape}  ({len(feature_names)} features)")
print(f"Attack rate:    {attack_labels.mean():.3f}")

In [ ]:
# Split the dataset
(
    training_features,
    test_features,
    training_attack_labels,
    test_attack_labels,
) = train_test_split(
    feature_matrix,
    attack_labels,
    test_size=CONFIG["test_ratio"],
    stratify=attack_labels,
    random_state=CONFIG["random_seed"],
)

# Train the model only on benign flows so labels do not influence fitting.
benign_training_features = training_features[training_attack_labels == 0]
print(
    "Fitting Isolation Forest on "
    f"{len(benign_training_features):,} benign flows..."
)
model = IsolationForest(
    n_estimators=CONFIG["n_estimators"],
    contamination=CONFIG["contamination"],
    random_state=CONFIG["random_seed"],
    n_jobs=-1,
)
model.fit(benign_training_features)
print("Done.")

In [ ]:
# Calibrate the anomaly scale using benign training scores only.
# The benign median maps to 0 and the benign 99th percentile maps to 1.
benign_raw_scores = -model.score_samples(benign_training_features)
score_lo = float(np.percentile(benign_raw_scores, 50))
score_hi = float(np.percentile(benign_raw_scores, 99))
print(f"score_lo (benign p50): {score_lo:.4f}")
print(f"score_hi (benign p99): {score_hi:.4f}")

def calculate_anomaly_probabilities(raw_scores):
    score_range = score_hi - score_lo
    if score_range == 0:
        return np.zeros_like(raw_scores)
    return np.clip(
        (raw_scores - score_lo) / score_range,
        0.0,
        1.0,
    )

In [ ]:
# Evaluate performance
test_raw_scores = -model.score_samples(test_features)
test_probabilities = calculate_anomaly_probabilities(test_raw_scores)

# Use unbounded scores for ROC-AUC so clipping does not create ranking ties.
roc_auc = roc_auc_score(test_attack_labels, test_raw_scores)
print(f"ROC-AUC: {roc_auc:.4f}\n")

predicted_attack_labels = (
    test_probabilities >= MEDIUM_SEVERITY_THRESHOLD
).astype(int)
print(classification_report(
    test_attack_labels,
    predicted_attack_labels,
    target_names=["benign", "attack"],
))
print("Confusion matrix:")
print(confusion_matrix(test_attack_labels, predicted_attack_labels))

# Compare thresholds to show the trade-off between precision and recall.
print(f"\n{'threshold':>10}{'precision':>12}{'recall':>10}{'f1':>10}")
for threshold in EVALUATION_THRESHOLDS:
    predicted_labels = (test_probabilities >= threshold).astype(int)
    precision = precision_score(
        test_attack_labels,
        predicted_labels,
        zero_division=0,
    )
    recall = recall_score(
        test_attack_labels,
        predicted_labels,
        zero_division=0,
    )
    f1 = f1_score(
        test_attack_labels,
        predicted_labels,
        zero_division=0,
    )
    print(
        f"{threshold:>10.2f}"
        f"{precision:>12.3f}"
        f"{recall:>10.3f}"
        f"{f1:>10.3f}"
    )

In [ ]:
# Save the model
model_bundle = {
    "model": model,
    "feature_names": feature_names,
    "score_lo": score_lo,
    "score_hi": score_hi,
    "model_version": "network-isolationforest-v1",
}
joblib.dump(model_bundle, CONFIG["output_path"])
print(f"Saved bundle to {CONFIG['output_path']}")

In [ ]:
# Verify the saved model
saved_bundle = joblib.load(CONFIG["output_path"])
saved_model = saved_bundle["model"]

def calculate_saved_probability(flow_features):
    raw_score = -saved_model.score_samples(
        flow_features.reshape(1, -1)
    )[0]
    score_range = saved_bundle["score_hi"] - saved_bundle["score_lo"]
    if score_range == 0:
        return 0.0
    return float(np.clip(
        (raw_score - saved_bundle["score_lo"]) / score_range,
        0.0,
        1.0,
    ))

benign_example = test_features[test_attack_labels == 0][0]
attack_example = test_features[test_attack_labels == 1][0]
benign_probability = calculate_saved_probability(benign_example)
attack_probability = calculate_saved_probability(attack_example)
print(f"Benign example anomaly probability: {benign_probability:.3f}")
print(f"Attack example anomaly probability: {attack_probability:.3f}")

assert roc_auc >= MINIMUM_EXPECTED_ROC_AUC, (
    f"ROC-AUC {roc_auc:.3f} too low - check the pipeline."
)
print("Sanity check passed.")

In [ ]:
# Copy the model to Google Drive
import os
import shutil

drive_model_path = (
    "/content/drive/MyDrive/cm3070_models/network_model.joblib"
)
os.makedirs(os.path.dirname(drive_model_path), exist_ok=True)
shutil.copy(CONFIG["output_path"], drive_model_path)

model_size_mb = os.path.getsize(drive_model_path) / 1024 / 1024
print(f"Saved to Drive: {drive_model_path}  ({model_size_mb:.1f} MB)")
print(
    "Download it, then place it at "
    "models/network_module/network_model.joblib"
)